# Retail Orders: Data Profile and Quality Checks

**Decision owner:** Head of Sales & Operations (ed-tech business)

**Decision supported:** Which products and cities to push, and whether the order data can be trusted for reporting.

This notebook checks the raw retail orders file against the data dictionary rules: completeness, uniqueness, validity, consistency and freshness. It ends with a failure summary and KPIs on the cleaned data.

**How to run:** add `retail-orders-raw.csv` as an input (Kaggle: Add Input, Colab: Files panel), then click Run All.

## 1. Load the data

In [19]:
import glob
import pandas as pd

paths = (glob.glob('/kaggle/input/**/retail-orders-raw.csv', recursive=True)
         + glob.glob('/content/**/retail-orders-raw.csv', recursive=True)
         + glob.glob('./**/retail-orders-raw.csv', recursive=True))
assert paths, 'File not found. Add retail-orders-raw.csv first, then run again.'
print('Using file:', paths[0])

df = pd.read_csv(paths[0])
print('Rows:', len(df), '| Columns:', df.shape[1])
df

Using file: /content/retail-orders-raw.csv
Rows: 12 | Columns: 9


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid


## 2. Rules from the data dictionary

| Column | Rule |
|---|---|
| order_id | Required and unique |
| order_date | Required, ISO format (YYYY-MM-DD), between 2025-01-01 and today |
| customer_segment | Student, Fresher or Professional (after normalization) |
| city | Required, non-empty text |
| category | Learning Kit, Course Access or Mentor Session |
| quantity | Whole number greater than zero |
| unit_price | Non-negative INR amount |
| discount_pct | Between 0 and 100; missing counts as zero only after justification |
| payment_status | Paid, Pending, Failed or Refunded (after normalization) |

## 3. Completeness: missing values

In [20]:
print(df.isnull().sum())
print()
print('Rows with at least one missing value:')
df[df.isnull().any(axis=1)]

order_id            0
order_date          1
customer_segment    0
city                1
category            0
quantity            0
unit_price          0
discount_pct        1
payment_status      0
dtype: int64

Rows with at least one missing value:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
11,RT-1011,NaN,Student,Kochi,Mentor Session,1,999,0.0,Paid


## 4. Uniqueness: duplicate order IDs

In [21]:
dups = df[df.duplicated('order_id', keep=False)]
print('Rows involved in duplicates:', len(dups))
dups

Rows involved in duplicates: 2


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid


## 5. Validity: values that break the rules

In [22]:
today = pd.Timestamp.today().normalize()

# Dates: strict YYYY-MM-DD, real calendar date, within the allowed range
df['date_clean'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce')
bad_date = (df['order_date'].isnull() | df['date_clean'].isnull()
            | (df['date_clean'] < '2025-01-01') | (df['date_clean'] > today))

# Quantity: whole number greater than zero
df['qty_num'] = pd.to_numeric(df['quantity'], errors='coerce')
bad_qty = df['qty_num'].isnull() | (df['qty_num'] <= 0) | (df['qty_num'] % 1 != 0)

# Discount: between 0 and 100 (missing values are covered by the completeness check)
bad_discount = df['discount_pct'].notnull() & ((df['discount_pct'] < 0) | (df['discount_pct'] > 100))

# Unit price: not negative
bad_price = df['unit_price'].isnull() | (df['unit_price'] < 0)

print('Invalid or missing dates:')
display(df.loc[bad_date, ['order_id', 'order_date']])
print('Invalid quantity:')
display(df.loc[bad_qty, ['order_id', 'quantity']])
print('Invalid discount:')
display(df.loc[bad_discount, ['order_id', 'discount_pct']])
print('Invalid unit price:')
display(df.loc[bad_price, ['order_id', 'unit_price']])

Invalid or missing dates:


,order_id,order_date
1,RT-1002,03/01/2026
6,RT-1006,2026-13-10
11,RT-1011,NaN


Invalid quantity:


,order_id,quantity
6,RT-1006,-1
8,RT-1008,two


Invalid discount:


,order_id,discount_pct
7,RT-1007,105.0


Invalid unit price:


,order_id,unit_price


## 6. Consistency: text formatting and allowed values

In [23]:
allowed = {
    'customer_segment': ['Student', 'Fresher', 'Professional'],
    'category': ['Learning Kit', 'Course Access', 'Mentor Session'],
    'payment_status': ['Paid', 'Pending', 'Failed', 'Refunded'],
}

inconsistent = pd.Series(False, index=df.index)
for col, valid in allowed.items():
    norm = df[col].astype(str).str.strip().str.title()
    inconsistent = inconsistent | (norm != df[col].astype(str))
    print(col, '- raw values:', df[col].value_counts().to_dict())
    print(col, '- values outside the allowed list after normalization:', int((~norm.isin(valid)).sum()))
    print()

print('Rows with inconsistent text formatting:')
df.loc[inconsistent, ['order_id', 'customer_segment', 'category', 'payment_status']]

customer_segment - raw values: {'Student': 4, 'Professional': 4, 'Fresher': 3, 'student': 1}
customer_segment - values outside the allowed list after normalization: 0

category - raw values: {'Learning Kit': 5, 'Course Access': 4, 'Mentor Session': 3}
category - values outside the allowed list after normalization: 0

payment_status - raw values: {'Paid': 7, 'Pending': 2, 'paid': 1, 'Failed': 1, 'Refunded': 1}
payment_status - values outside the allowed list after normalization: 0

Rows with inconsistent text formatting:


,order_id,customer_segment,category,payment_status
1,RT-1002,Fresher,Course Access,paid
2,RT-1003,student,Course Access,Pending


## 7. Freshness: how recent is the data?

In [24]:
latest = df['date_clean'].max()
age_days = (today - latest).days
print('Latest valid order date:', latest.date())
print('Today:', today.date())
print('Data age in days:', age_days)

Latest valid order date: 2026-01-18
Today: 2026-10-31
Data age in days: 286


## 8. Summary of failures

In [25]:
total = len(df)
missing_date = df['order_date'].isnull()
missing_city = df['city'].isnull() | (df['city'].astype(str).str.strip() == '')
missing_disc = df['discount_pct'].isnull()
dup_rows = df.duplicated('order_id', keep='first')

checks = [
    ('Completeness', 'Missing order_date', int(missing_date.sum())),
    ('Completeness', 'Missing city', int(missing_city.sum())),
    ('Completeness', 'Missing discount_pct', int(missing_disc.sum())),
    ('Uniqueness', 'Duplicate order_id (extra copies)', int(dup_rows.sum())),
    ('Validity', 'Invalid order_date (format, range or missing)', int(bad_date.sum())),
    ('Validity', 'Invalid quantity', int(bad_qty.sum())),
    ('Validity', 'Discount outside 0-100', int(bad_discount.sum())),
    ('Validity', 'Invalid unit_price', int(bad_price.sum())),
    ('Consistency', 'Inconsistent text case', int(inconsistent.sum())),
    ('Freshness', 'Latest data older than 30 days', int(age_days > 30)),
]
summary = pd.DataFrame(checks, columns=['dimension', 'check', 'failing_rows'])
summary['failure_pct'] = (summary['failing_rows'] / total * 100).round(1)
summary
summary.loc[summary['dimension'] == 'Freshness', 'failure_pct'] = None

## 9. Clean the data and compute KPIs

A row is analysis-ready only if it passes every rule. Failing rows are excluded from KPIs and listed so the data owner can fix them at the source. Missing discounts are not treated as zero, because there is no documented justification.

In [26]:
clean = df.copy()
for col in ['customer_segment', 'category', 'payment_status']:
    clean[col] = clean[col].astype(str).str.strip().str.title()

clean = clean[~clean.duplicated('order_id', keep='first')]

bad = (clean['date_clean'].isnull()
       | clean['city'].isnull()
       | clean['discount_pct'].isnull()
       | (clean['discount_pct'] < 0) | (clean['discount_pct'] > 100)
       | clean['qty_num'].isnull() | (clean['qty_num'] <= 0)
       | (clean['unit_price'] < 0))
rejected = clean[bad]
clean = clean[~bad].copy()

print('Rows in raw file:', len(df))
print('Duplicate copies removed:', int(dup_rows.sum()))
print('Rows rejected for rule failures:', len(rejected))
print('Rows analysis-ready:', len(clean))
print()
print('Rejected rows:')
display(rejected[['order_id', 'order_date', 'city', 'quantity', 'discount_pct']])

clean['gross_revenue'] = clean['qty_num'] * clean['unit_price']
clean['net_revenue'] = clean['gross_revenue'] * (1 - clean['discount_pct'] / 100)
clean

Rows in raw file: 12
Duplicate copies removed: 1
Rows rejected for rule failures: 7
Rows analysis-ready: 4

Rejected rows:


,order_id,order_date,city,quantity,discount_pct
1,RT-1002,03/01/2026,Bengaluru,1,0.0
2,RT-1003,2026-01-05,Chennai,1,NaN
5,RT-1005,2026-01-09,NaN,1,0.0
6,RT-1006,2026-13-10,Pune,-1,10.0
7,RT-1007,2026-01-12,Mumbai,2,105.0
8,RT-1008,2026-01-14,Bengaluru,two,0.0
11,RT-1011,NaN,Kochi,1,0.0


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,date_clean,qty_num,gross_revenue,net_revenue
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid,2026-01-03,2.0,1598.0,1438.20
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2026-01-07,3.0,2397.0,2277.15
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid,2026-01-16,1.0,799.0,799.00
10,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15.0,Refunded,2026-01-18,2.0,2998.0,2548.30


In [27]:
paid = clean[clean['payment_status'] == 'Paid']

kpis = pd.DataFrame([
    ['Gross revenue (INR)', clean['gross_revenue'].sum()],
    ['Net revenue, paid orders (INR)', paid['net_revenue'].sum()],
    ['Total orders', clean['order_id'].nunique()],
    ['Units sold, paid orders', paid['qty_num'].sum()],
    ['Average order value (INR)', paid['net_revenue'].sum() / max(len(paid), 1)],
    ['Payment success rate %', round((clean['payment_status'] == 'Paid').mean() * 100, 1)],
    ['Failed payment rate %', round((clean['payment_status'] == 'Failed').mean() * 100, 1)],
    ['Refund rate %', round((clean['payment_status'] == 'Refunded').mean() * 100, 1)],
], columns=['KPI', 'Value'])
display(kpis)

print('Net revenue by category (paid orders):')
display(paid.groupby('category')['net_revenue'].sum().sort_values(ascending=False))

,KPI,Value
0,Gross revenue (INR),7792.000000
1,"Net revenue, paid orders (INR)",4514.350000
2,Total orders,4.000000
3,"Units sold, paid orders",6.000000
4,Average order value (INR),1504.783333
5,Payment success rate %,75.000000
6,Failed payment rate %,0.000000
7,Refund rate %,25.000000


Net revenue by category (paid orders):


,net_revenue
category,
Learning Kit,4514.35


## 10. Observations

- Only a small share of the raw rows pass every rule, so reporting on this file as it is would be unreliable.
- The latest valid order is months old, so the data fails the freshness expectation.
- Duplicate and invalid rows should be fixed at the source before any dashboard refresh.

Thresholds and escalation actions are in the data quality contract.